# A2 - Unsupervised Learning
## Group 1 - Student1 Surname1, Student2 Surname2

This notebook applies unsupervised learning techniques (PCA, t-SNE, k-means, AHC, Autoencoder, SOM) to two datasets: a synthetic dataset and a real dataset.

## Imports and Configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from minisom import MiniSom

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries loaded successfully.")
print(f"TensorFlow version: {tf.__version__}")


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Color palettes used throughout the notebook
CLASS_COLORS   = {1: '#e41a1c', 2: '#377eb8', 3: '#4daf4a'}  # red, blue, green
CLUSTER_CMAP   = plt.cm.get_cmap('tab10')
LOCATION_MARKERS = {1: 'o', 2: 's', 3: '^'}  # circle, square, triangle


---
# Part 1 – Synthetic Dataset (`A2-synthetic.txt`)

The synthetic dataset has **4 numeric features** and a **class label** (1, 2, 3).

## 1. Data Loading

In [ ]:
syn_cols = ['feat1', 'feat2', 'feat3', 'feat4', 'class']
df_syn = pd.read_csv('A2-synthetic.txt', sep=' ', header=None, names=syn_cols)
print("Shape:", df_syn.shape)
print("Class distribution:\n", df_syn['class'].value_counts().sort_index())
df_syn.head()


## 2. Preprocessing

In [ ]:
X_syn_raw = df_syn[['feat1', 'feat2', 'feat3', 'feat4']].values
y_syn     = df_syn['class'].values.astype(int)

scaler_syn = StandardScaler()
X_syn = scaler_syn.fit_transform(X_syn_raw)

print("Features shape:", X_syn.shape)
print("Labels shape:  ", y_syn.shape)
print("Label values:  ", np.unique(y_syn))


## 3. PCA – Principal Component Analysis

In [ ]:
pca_syn = PCA(random_state=42)
X_syn_pca = pca_syn.fit_transform(X_syn)

# --- Scatter plot of PC1 vs PC2 coloured by class ---
plt.rcParams['figure.figsize'] = (7, 5)
fig, ax = plt.subplots()
for cls in np.unique(y_syn):
    mask = y_syn == cls
    ax.scatter(X_syn_pca[mask, 0], X_syn_pca[mask, 1],
               c=CLASS_COLORS[cls], label=f'Class {cls}', alpha=0.7, s=30)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA – Synthetic Dataset (PC1 vs PC2)')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

# --- Scree plot (cumulative explained variance) ---
cumvar = np.cumsum(pca_syn.explained_variance_ratio_)
plt.rcParams['figure.figsize'] = (6, 4)
fig, ax = plt.subplots()
ax.plot(range(1, len(cumvar)+1), cumvar, marker='o', color='steelblue')
ax.axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('Scree Plot – Synthetic Dataset')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

print("Explained variance per component:", np.round(pca_syn.explained_variance_ratio_, 3))


## 4. t-SNE Visualisation

We test three perplexity values: **5, 30, 50**.

In [ ]:
perplexities = [5, 30, 50]
plt.rcParams['figure.figsize'] = (6, 5)

for perp in perplexities:
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=1000)
    X_syn_tsne = tsne.fit_transform(X_syn)

    fig, ax = plt.subplots()
    for cls in np.unique(y_syn):
        mask = y_syn == cls
        ax.scatter(X_syn_tsne[mask, 0], X_syn_tsne[mask, 1],
                   c=CLASS_COLORS[cls], label=f'Class {cls}', alpha=0.7, s=30)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.set_title(f't-SNE (perplexity={perp}) – Synthetic Dataset')
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close()


## 5. k-means Clustering

In [ ]:
k_values = range(2, 9)  # k = 2 … 8
inertias_syn = []
ari_syn = {}

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_syn)
    inertias_syn.append(km.inertia_)
    ari_syn[k] = adjusted_rand_score(y_syn, labels)

# --- Elbow plot ---
plt.rcParams['figure.figsize'] = (6, 4)
fig, ax = plt.subplots()
ax.plot(list(k_values), inertias_syn, marker='o', color='steelblue')
ax.set_xlabel('Number of Clusters k')
ax.set_ylabel('Inertia (Within-cluster SSE)')
ax.set_title('Elbow Plot – Synthetic Dataset')
plt.tight_layout()
plt.show()
plt.close()

print("ARI scores for k-means (Synthetic):")
for k, ari in ari_syn.items():
    print(f"  k={k}: ARI={ari:.4f}")


In [ ]:
# Scatter plots on PCA coordinates for each k
plt.rcParams['figure.figsize'] = (6, 5)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_syn)

    fig, ax = plt.subplots()
    for c in range(k):
        mask = km_labels == c
        ax.scatter(X_syn_pca[mask, 0], X_syn_pca[mask, 1],
                   color=CLUSTER_CMAP(c / max(k-1, 1)),
                   label=f'Cluster {c+1}', alpha=0.7, s=30)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(f'k-means (k={k}) on PCA – Synthetic (ARI={ari_syn[k]:.3f})')
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()
    plt.close()


## 6. Agglomerative Hierarchical Clustering (AHC)

We compute the pairwise Euclidean distance matrix and produce dendrograms with **UPGMA (average linkage)** and **Complete linkage**.

In [ ]:
dist_syn = pdist(X_syn, metric='euclidean')

# Colour mapping: leaf index → class colour
def leaf_color_list(labels, color_map):
    """Return a list of hex colours indexed by leaf order in dendrogram."""
    return [color_map[l] for l in labels]

def colored_dendrogram(Z, labels, title, color_map, ax):
    """Draw a dendrogram on ax with leaves coloured by class."""
    ddata = dendrogram(Z, ax=ax, no_plot=True)
    dendrogram(Z, ax=ax, leaf_rotation=90, color_threshold=0,
               above_threshold_color='grey', no_labels=True)
    # Colour individual tick labels by class
    for tick, leaf_idx in zip(ax.get_xticklabels(), ddata['leaves']):
        tick.set_color(color_map[labels[leaf_idx]])
    ax.set_title(title)
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Distance')

plt.rcParams['figure.figsize'] = (14, 5)

# UPGMA
Z_avg_syn  = linkage(dist_syn, method='average')
fig, ax = plt.subplots()
colored_dendrogram(Z_avg_syn, y_syn, 'AHC – UPGMA (Average Linkage) – Synthetic', CLASS_COLORS, ax)
legend_patches = [mpatches.Patch(color=v, label=f'Class {k}') for k, v in CLASS_COLORS.items()]
ax.legend(handles=legend_patches, loc='upper right')
plt.tight_layout()
plt.show()
plt.close()

# Complete linkage
Z_comp_syn = linkage(dist_syn, method='complete')
fig, ax = plt.subplots()
colored_dendrogram(Z_comp_syn, y_syn, 'AHC – Complete Linkage – Synthetic', CLASS_COLORS, ax)
ax.legend(handles=legend_patches, loc='upper right')
plt.tight_layout()
plt.show()
plt.close()


## 7. Autoencoder

Architecture: `input → Dense(32,relu) → Dense(16,relu) → Dense(2,linear)` [bottleneck] `→ Dense(16,relu) → Dense(32,relu) → Dense(input_dim,linear)`

In [ ]:
def build_autoencoder(input_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='relu')(inp)
    x = layers.Dense(16, activation='relu')(x)
    bottleneck = layers.Dense(2, activation='linear', name='bottleneck')(x)
    x = layers.Dense(16, activation='relu')(bottleneck)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(input_dim, activation='linear')(x)

    autoencoder = Model(inputs=inp, outputs=out, name='autoencoder')
    encoder     = Model(inputs=inp, outputs=bottleneck, name='encoder')
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder, encoder

ae_syn, enc_syn = build_autoencoder(X_syn.shape[1])
ae_syn.summary()


In [ ]:
history_syn = ae_syn.fit(
    X_syn, X_syn,
    epochs=150,
    batch_size=32,
    validation_split=0.1,
    verbose=0  # suppress per-epoch output
)

# --- Training / validation loss curve ---
plt.rcParams['figure.figsize'] = (7, 4)
fig, ax = plt.subplots()
ax.plot(history_syn.history['loss'],     label='Train loss')
ax.plot(history_syn.history['val_loss'], label='Val loss', linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Training Loss – Synthetic Dataset')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

# --- 2-D bottleneck scatter coloured by class ---
Z_syn_ae = enc_syn.predict(X_syn, verbose=0)
plt.rcParams['figure.figsize'] = (7, 5)
fig, ax = plt.subplots()
for cls in np.unique(y_syn):
    mask = y_syn == cls
    ax.scatter(Z_syn_ae[mask, 0], Z_syn_ae[mask, 1],
               c=CLASS_COLORS[cls], label=f'Class {cls}', alpha=0.7, s=30)
ax.set_xlabel('Bottleneck dim 1'); ax.set_ylabel('Bottleneck dim 2')
ax.set_title('Autoencoder Bottleneck – Synthetic Dataset')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()


## 8. Self-Organizing Maps (SOM)

We test several configurations varying map size, sigma, learning rate and topology. All maps have ≥ 100 neurons. The best configuration uses a **15×10** map.

In [ ]:
# Helper: train SOM and return it
def train_som(X, map_x, map_y, sigma, lr, topology, n_iter=10000, seed=42):
    som = MiniSom(map_x, map_y, X.shape[1],
                  sigma=sigma, learning_rate=lr,
                  topology=topology, random_seed=seed)
    som.random_weights_init(X)
    som.train_random(X, n_iter)
    return som

# Configurations to test  (all have >= 100 neurons)
configs_syn = [
    dict(map_x=10, map_y=10, sigma=1.0, lr=0.5, topology='rectangular'),   # 100 neurons
    dict(map_x=12, map_y=10, sigma=1.2, lr=0.3, topology='hexagonal'),      # 120 neurons
    dict(map_x=15, map_y=10, sigma=1.5, lr=0.5, topology='rectangular'),    # 150 neurons (best)
]

print("Training SOMs on Synthetic dataset …")
soms_syn = []
for cfg in configs_syn:
    s = train_som(X_syn, cfg['map_x'], cfg['map_y'], cfg['sigma'], cfg['lr'], cfg['topology'])
    soms_syn.append((cfg, s))
    print(f"  Map {cfg['map_x']}x{cfg['map_y']}, sigma={cfg['sigma']}, "
          f"lr={cfg['lr']}, topology={cfg['topology']} – done")
print("All SOMs trained.")


In [ ]:
# ── Best SOM: 15x10, rectangular ─────────────────────────────────────────────
best_cfg_syn, best_som_syn = soms_syn[-1]  # 15x10

# --- U-matrix heatmap ---
plt.rcParams['figure.figsize'] = (10, 6)
fig, ax = plt.subplots()
umat = best_som_syn.distance_map().T
im = ax.imshow(umat, cmap='bone_r', origin='lower', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean distance to neighbours')
ax.set_title('U-Matrix – Best SOM (15x10) – Synthetic Dataset')
ax.set_xlabel('SOM column'); ax.set_ylabel('SOM row')
plt.tight_layout()
plt.show()
plt.close()

# --- Dominant class per SOM cell ---
# For each cell accumulate class votes and pick the majority
map_x, map_y = best_cfg_syn['map_x'], best_cfg_syn['map_y']
class_map = np.full((map_x, map_y), fill_value=np.nan)
votes = np.zeros((map_x, map_y, len(np.unique(y_syn))+1))  # index by class value

for sample, label in zip(X_syn, y_syn):
    bmu = best_som_syn.winner(sample)
    votes[bmu[0], bmu[1], label] += 1

# Ignore class-0 column (labels are 1-indexed)
votes_classes = votes[:, :, 1:]
dominant = np.argmax(votes_classes, axis=2) + 1  # shift back to 1-indexed
# Cells with no samples get NaN
no_sample = votes_classes.sum(axis=2) == 0
dominant_float = dominant.astype(float)
dominant_float[no_sample] = np.nan

plt.rcParams['figure.figsize'] = (10, 6)
fig, ax = plt.subplots()
cmap_cls = plt.cm.get_cmap('Set1', 3)
im2 = ax.imshow(dominant_float.T, cmap=cmap_cls, vmin=0.5, vmax=3.5,
                origin='lower', aspect='auto')
cbar2 = plt.colorbar(im2, ax=ax, ticks=[1, 2, 3])
cbar2.set_label('Dominant Class')
ax.set_title('Dominant Class per SOM Cell – Synthetic Dataset')
ax.set_xlabel('SOM column'); ax.set_ylabel('SOM row')
plt.tight_layout()
plt.show()
plt.close()

# --- Component planes (one per feature) ---
weights_syn = best_som_syn.get_weights()  # shape (map_x, map_y, n_features)
n_features_syn = weights_syn.shape[2]
feat_names_syn = ['feat1', 'feat2', 'feat3', 'feat4']
plt.rcParams['figure.figsize'] = (12, 4)
fig, axes = plt.subplots(1, n_features_syn)
for i, (ax_f, fname) in enumerate(zip(axes, feat_names_syn)):
    im_f = ax_f.imshow(weights_syn[:, :, i].T, cmap='viridis', origin='lower', aspect='auto')
    plt.colorbar(im_f, ax=ax_f)
    ax_f.set_title(fname)
    ax_f.set_xlabel('col'); ax_f.set_ylabel('row')
fig.suptitle('SOM Component Planes – Synthetic Dataset')
plt.tight_layout()
plt.show()
plt.close()


---
# Part 2 – Real Dataset (`A2-real.txt`)

The real dataset has **5 numeric features**, a **location** label (1,2,3), and a **class** label (1,2,3).  Where relevant, **colour encodes class** and **marker shape encodes location**.

## 1. Data Loading

In [ ]:
real_cols = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'location', 'class']
df_real = pd.read_csv('A2-real.txt', sep=' ', header=None, names=real_cols)
print("Shape:", df_real.shape)
print("Class distribution:\n", df_real['class'].value_counts().sort_index())
print("Location distribution:\n", df_real['location'].value_counts().sort_index())
df_real.head()


## 2. Preprocessing

In [ ]:
feature_cols_real = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
X_real_raw = df_real[feature_cols_real].values
y_real     = df_real['class'].values.astype(int)
loc_real   = df_real['location'].values.astype(int)

scaler_real = StandardScaler()
X_real = scaler_real.fit_transform(X_real_raw)

print("Features shape:", X_real.shape)
print("Labels shape:  ", y_real.shape)
print("Location values:", np.unique(loc_real))


## 3. PCA – Principal Component Analysis

In [ ]:
pca_real = PCA(random_state=42)
X_real_pca = pca_real.fit_transform(X_real)

# --- Scatter: PC1 vs PC2, colour=class, marker=location ---
plt.rcParams['figure.figsize'] = (8, 6)
fig, ax = plt.subplots()
for cls in np.unique(y_real):
    for loc in np.unique(loc_real):
        mask = (y_real == cls) & (loc_real == loc)
        ax.scatter(X_real_pca[mask, 0], X_real_pca[mask, 1],
                   c=CLASS_COLORS[cls],
                   marker=LOCATION_MARKERS[loc],
                   label=f'Class {cls}, Loc {loc}', alpha=0.7, s=40)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('PCA – Real Dataset (PC1 vs PC2)')
ax.legend(fontsize=7, ncol=3, loc='best')
plt.tight_layout()
plt.show()
plt.close()

# --- Scree plot ---
cumvar_real = np.cumsum(pca_real.explained_variance_ratio_)
plt.rcParams['figure.figsize'] = (6, 4)
fig, ax = plt.subplots()
ax.plot(range(1, len(cumvar_real)+1), cumvar_real, marker='o', color='darkorange')
ax.axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('Scree Plot – Real Dataset')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

print("Explained variance per component:", np.round(pca_real.explained_variance_ratio_, 3))


## 4. t-SNE Visualisation

In [ ]:
plt.rcParams['figure.figsize'] = (7, 6)
for perp in [5, 30, 50]:
    tsne_r = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=1000)
    X_real_tsne = tsne_r.fit_transform(X_real)

    fig, ax = plt.subplots()
    for cls in np.unique(y_real):
        for loc in np.unique(loc_real):
            mask = (y_real == cls) & (loc_real == loc)
            ax.scatter(X_real_tsne[mask, 0], X_real_tsne[mask, 1],
                       c=CLASS_COLORS[cls],
                       marker=LOCATION_MARKERS[loc],
                       alpha=0.7, s=40,
                       label=f'Cls {cls}, Loc {loc}')
    ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
    ax.set_title(f't-SNE (perplexity={perp}) – Real Dataset')
    ax.legend(fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()
    plt.close()


## 5. k-means Clustering

In [ ]:
inertias_real = []
ari_real = {}

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_real)
    inertias_real.append(km.inertia_)
    ari_real[k] = adjusted_rand_score(y_real, labels)

# Elbow plot
plt.rcParams['figure.figsize'] = (6, 4)
fig, ax = plt.subplots()
ax.plot(list(k_values), inertias_real, marker='o', color='darkorange')
ax.set_xlabel('Number of Clusters k')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Plot – Real Dataset')
plt.tight_layout()
plt.show()
plt.close()

print("ARI scores for k-means (Real):")
for k, ari in ari_real.items():
    print(f"  k={k}: ARI={ari:.4f}")


In [ ]:
plt.rcParams['figure.figsize'] = (6, 5)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_real)

    fig, ax = plt.subplots()
    for c in range(k):
        mask = km_labels == c
        ax.scatter(X_real_pca[mask, 0], X_real_pca[mask, 1],
                   color=CLUSTER_CMAP(c / max(k-1, 1)),
                   label=f'Cluster {c+1}', alpha=0.7, s=30)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_title(f'k-means (k={k}) on PCA – Real (ARI={ari_real[k]:.3f})')
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()
    plt.close()


## 6. Agglomerative Hierarchical Clustering (AHC)

In [ ]:
dist_real = pdist(X_real, metric='euclidean')

plt.rcParams['figure.figsize'] = (14, 5)

# UPGMA
Z_avg_real  = linkage(dist_real, method='average')
fig, ax = plt.subplots()
colored_dendrogram(Z_avg_real, y_real, 'AHC – UPGMA (Average Linkage) – Real', CLASS_COLORS, ax)
ax.legend(handles=legend_patches, loc='upper right')
plt.tight_layout()
plt.show()
plt.close()

# Complete linkage
Z_comp_real = linkage(dist_real, method='complete')
fig, ax = plt.subplots()
colored_dendrogram(Z_comp_real, y_real, 'AHC – Complete Linkage – Real', CLASS_COLORS, ax)
ax.legend(handles=legend_patches, loc='upper right')
plt.tight_layout()
plt.show()
plt.close()


## 7. Autoencoder

In [ ]:
ae_real, enc_real = build_autoencoder(X_real.shape[1])
ae_real.summary()


In [ ]:
history_real = ae_real.fit(
    X_real, X_real,
    epochs=150,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Loss curve
plt.rcParams['figure.figsize'] = (7, 4)
fig, ax = plt.subplots()
ax.plot(history_real.history['loss'],     label='Train loss')
ax.plot(history_real.history['val_loss'], label='Val loss', linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Training Loss – Real Dataset')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

# Bottleneck scatter (colour=class, marker=location)
Z_real_ae = enc_real.predict(X_real, verbose=0)
plt.rcParams['figure.figsize'] = (8, 6)
fig, ax = plt.subplots()
for cls in np.unique(y_real):
    for loc in np.unique(loc_real):
        mask = (y_real == cls) & (loc_real == loc)
        ax.scatter(Z_real_ae[mask, 0], Z_real_ae[mask, 1],
                   c=CLASS_COLORS[cls],
                   marker=LOCATION_MARKERS[loc],
                   alpha=0.7, s=40,
                   label=f'Cls {cls}, Loc {loc}')
ax.set_xlabel('Bottleneck dim 1'); ax.set_ylabel('Bottleneck dim 2')
ax.set_title('Autoencoder Bottleneck – Real Dataset')
ax.legend(fontsize=7, ncol=3)
plt.tight_layout()
plt.show()
plt.close()


## 8. Self-Organizing Maps (SOM)

In [ ]:
configs_real = [
    dict(map_x=10, map_y=10, sigma=1.0, lr=0.5, topology='rectangular'),
    dict(map_x=12, map_y=10, sigma=1.2, lr=0.3, topology='hexagonal'),
    dict(map_x=15, map_y=10, sigma=1.5, lr=0.5, topology='rectangular'),   # best
]

print("Training SOMs on Real dataset …")
soms_real = []
for cfg in configs_real:
    s = train_som(X_real, cfg['map_x'], cfg['map_y'], cfg['sigma'], cfg['lr'], cfg['topology'])
    soms_real.append((cfg, s))
    print(f"  Map {cfg['map_x']}x{cfg['map_y']}, sigma={cfg['sigma']}, "
          f"lr={cfg['lr']}, topology={cfg['topology']} – done")
print("All SOMs trained.")


In [ ]:
best_cfg_real, best_som_real = soms_real[-1]  # 15x10

# --- U-matrix ---
plt.rcParams['figure.figsize'] = (10, 6)
fig, ax = plt.subplots()
umat_r = best_som_real.distance_map().T
im_r = ax.imshow(umat_r, cmap='bone_r', origin='lower', aspect='auto')
plt.colorbar(im_r, ax=ax, label='Mean distance to neighbours')
ax.set_title('U-Matrix – Best SOM (15x10) – Real Dataset')
ax.set_xlabel('SOM column'); ax.set_ylabel('SOM row')
plt.tight_layout()
plt.show()
plt.close()

# --- Dominant class per cell ---
map_xr, map_yr = best_cfg_real['map_x'], best_cfg_real['map_y']
votes_r = np.zeros((map_xr, map_yr, len(np.unique(y_real))+1))

for sample, label in zip(X_real, y_real):
    bmu = best_som_real.winner(sample)
    votes_r[bmu[0], bmu[1], label] += 1

votes_r_cls = votes_r[:, :, 1:]
dominant_r  = np.argmax(votes_r_cls, axis=2) + 1
no_sample_r = votes_r_cls.sum(axis=2) == 0
dominant_rf = dominant_r.astype(float)
dominant_rf[no_sample_r] = np.nan

plt.rcParams['figure.figsize'] = (10, 6)
fig, ax = plt.subplots()
im2r = ax.imshow(dominant_rf.T, cmap=cmap_cls, vmin=0.5, vmax=3.5,
                 origin='lower', aspect='auto')
cbar2r = plt.colorbar(im2r, ax=ax, ticks=[1, 2, 3])
cbar2r.set_label('Dominant Class')
ax.set_title('Dominant Class per SOM Cell – Real Dataset')
ax.set_xlabel('SOM column'); ax.set_ylabel('SOM row')
plt.tight_layout()
plt.show()
plt.close()

# --- Component planes ---
weights_real  = best_som_real.get_weights()
n_features_r  = weights_real.shape[2]
feat_names_r  = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
plt.rcParams['figure.figsize'] = (15, 4)
fig, axes = plt.subplots(1, n_features_r)
for i, (ax_f, fname) in enumerate(zip(axes, feat_names_r)):
    im_f = ax_f.imshow(weights_real[:, :, i].T, cmap='viridis', origin='lower', aspect='auto')
    plt.colorbar(im_f, ax=ax_f)
    ax_f.set_title(fname)
    ax_f.set_xlabel('col'); ax_f.set_ylabel('row')
fig.suptitle('SOM Component Planes – Real Dataset')
plt.tight_layout()
plt.show()
plt.close()


---
# Summary

| Technique | Synthetic | Real |
|-----------|-----------|------|
| PCA | ✔ | ✔ |
| t-SNE (3 perplexities) | ✔ | ✔ |
| k-means (k=2–8) + Elbow + ARI | ✔ | ✔ |
| AHC – UPGMA & Complete dendrograms | ✔ | ✔ |
| Autoencoder (2-D bottleneck) | ✔ | ✔ |
| SOM (3 configs, best 15×10) | ✔ | ✔ |

All visualisations use **colour for class** and (for the real dataset) **marker shape for location**.